## 1. Import the libraries
These are standard Python libraries for calculations, tables and graphs.

In [ ]:
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

## 2. Set up the household

Every appliance has a power rating, a run duration and a list of hours when it is allowed to start. For example, the air conditioner can only start between 10:00 and 20:00.

The scheduler assumes each appliance runs once per day for its full duration.

In [ ]:
hours = list(range(24))

# Example electricity price for each hour of the day.
price = [
    5.0, 4.8, 4.6, 4.5, 4.5, 4.8,
    5.5, 7.0, 8.0, 8.5, 8.0, 7.5,
    7.0, 6.8, 6.5, 7.0, 8.0, 9.0,
    9.5, 9.0, 8.0, 7.0, 6.0, 5.5
]

appliances = [
    {"name": "Washing machine", "power": 0.7, "duration": 2, "start_hours": list(range(0, 22))},
    {"name": "Dishwasher",      "power": 1.2, "duration": 2, "start_hours": list(range(0, 22))},
    {"name": "EV charger",      "power": 3.0, "duration": 3, "start_hours": list(range(0, 22))},
    {"name": "Water heater",    "power": 1.5, "duration": 2, "start_hours": list(range(0, 22))},
    {"name": "Air conditioner", "power": 1.8, "duration": 4, "start_hours": list(range(10, 21))},
    {"name": "Clothes dryer",   "power": 2.0, "duration": 2, "start_hours": list(range(0, 22))}
]

max_load = 5.0  # Maximum combined appliance power in any hour

plt.figure(figsize=(10, 4))
plt.plot(hours, price, marker="o")
plt.xticks(range(0, 24, 2))
plt.xlabel("Hour of day")
plt.ylabel("Example price per kWh")
plt.title("Electricity price during the day")
plt.tight_layout()
plt.show()

## 3. Cost and constraint functions

The cost of running an appliance is the sum of `power × price` for each hour it is running.

A schedule is valid when:
1. Every appliance starts in one of its allowed hours.
2. The appliance finishes before the day ends.
3. The combined load never goes above `max_load`.

During optimization, invalid schedules receive a large penalty. We still check feasibility separately before calling a schedule valid.

In [ ]:
def appliance_cost(appliance, start):
    total = 0
    for h in range(start, start + appliance["duration"]):
        total += appliance["power"] * price[h]
    return total


def load_by_hour(schedule):
    load = [0.0] * 24

    for appliance, start in zip(appliances, schedule):
        for h in range(start, start + appliance["duration"]):
            load[h] += appliance["power"]

    return load


def is_valid(schedule):
    for appliance, start in zip(appliances, schedule):
        if start not in appliance["start_hours"]:
            return False
        if start + appliance["duration"] > 24:
            return False

    return max(load_by_hour(schedule)) <= max_load


def schedule_cost(schedule):
    # First calculate the real electricity cost.
    total = sum(
        appliance_cost(appliance, start)
        for appliance, start in zip(appliances, schedule)
    )

    # Add a penalty if the schedule exceeds the household load limit.
    load = load_by_hour(schedule)
    overload = sum(max(0, value - max_load) for value in load)
    total += overload * 1000

    # Penalize any appliance that starts outside its allowed window.
    for appliance, start in zip(appliances, schedule):
        if start not in appliance["start_hours"]:
            total += 1000

    return total


def random_schedule():
    return [
        random.choice(appliance["start_hours"])
        for appliance in appliances
    ]


def change_one_start(schedule):
    # Make a copy, then move one appliance to a different allowed hour.
    new_schedule = schedule.copy()
    i = random.randrange(len(appliances))
    new_schedule[i] = random.choice(appliances[i]["start_hours"])
    return new_schedule


def show_schedule(schedule):
    rows = []
    for appliance, start in zip(appliances, schedule):
        end = start + appliance["duration"]
        rows.append({
            "Appliance": appliance["name"],
            "Start": f"{start:02d}:00",
            "Finish": f"{end:02d}:00",
            "Power (kW)": appliance["power"],
            "Duration (hours)": appliance["duration"],
            "Cost": round(appliance_cost(appliance, start), 2)
        })
    return pd.DataFrame(rows)

## 4. Create a simple baseline

The baseline starts each appliance at the first allowed hour. We use it as a comparison, not as the optimized answer.

In [ ]:
baseline = [appliance["start_hours"][0] for appliance in appliances]
baseline_real_cost = sum(
    appliance_cost(appliance, start)
    for appliance, start in zip(appliances, baseline)
)

print("Baseline schedule")
display(show_schedule(baseline))
print("Baseline electricity cost:", round(baseline_real_cost, 2))
print("Baseline valid:", is_valid(baseline))

## 5. Hill Climbing

Hill Climbing starts with a random schedule. It tries moving one appliance. If the new schedule has a lower score, it keeps the change. Repeating this process improves the answer, although it can get stuck in a local optimum.

In [ ]:
def hill_climbing(steps=2500, restarts=8):
    best_schedule = None
    best_score = float("inf")
    history = []

    for _ in range(restarts):
        current = random_schedule()
        current_score = schedule_cost(current)

        for _ in range(steps):
            candidate = change_one_start(current)
            candidate_score = schedule_cost(candidate)

            if candidate_score < current_score:
                current = candidate
                current_score = candidate_score

            if current_score < best_score:
                best_schedule = current.copy()
                best_score = current_score

            history.append(best_score)

    return best_schedule, best_score, history


hc_schedule, hc_score, hc_history = hill_climbing()
print("Hill Climbing cost score:", round(hc_score, 2))
print("Valid schedule:", is_valid(hc_schedule))
display(show_schedule(hc_schedule))

plt.figure(figsize=(9, 4))
plt.plot(hc_history)
plt.xlabel("Search step")
plt.ylabel("Best score so far")
plt.title("Hill Climbing progress")
plt.tight_layout()
plt.show()

## 6. Simulated Annealing

Unlike Hill Climbing, Simulated Annealing can sometimes accept a worse move. This helps it explore different parts of the search space early on. As the temperature falls, it becomes less likely to accept worse moves.

In [ ]:
def simulated_annealing(steps=4000, temperature=100, cooling=0.995):
    current = random_schedule()
    current_score = schedule_cost(current)
    best_schedule = current.copy()
    best_score = current_score
    history = []

    for _ in range(steps):
        candidate = change_one_start(current)
        candidate_score = schedule_cost(candidate)
        difference = candidate_score - current_score

        # Always accept a better move; sometimes accept a worse one.
        if difference < 0 or random.random() < math.exp(-difference / max(temperature, 1e-9)):
            current = candidate
            current_score = candidate_score

        if current_score < best_score:
            best_schedule = current.copy()
            best_score = current_score

        history.append(best_score)
        temperature *= cooling

    return best_schedule, best_score, history


sa_schedule, sa_score, sa_history = simulated_annealing()
print("Simulated Annealing cost score:", round(sa_score, 2))
print("Valid schedule:", is_valid(sa_schedule))
display(show_schedule(sa_schedule))

plt.figure(figsize=(9, 4))
plt.plot(sa_history)
plt.xlabel("Iteration")
plt.ylabel("Best score so far")
plt.title("Simulated Annealing progress")
plt.tight_layout()
plt.show()

## 7. Genetic Algorithm

A schedule is treated like a chromosome: each position stores one appliance's start hour. The algorithm keeps better schedules, combines two parent schedules, and occasionally changes a start hour through mutation.

In [ ]:
def genetic_algorithm(population_size=60, generations=180, mutation_rate=0.2):
    population = [random_schedule() for _ in range(population_size)]
    history = []

    def choose_parent():
        # Tournament selection: choose the best of a few random candidates.
        group = random.sample(population, 3)
        return min(group, key=schedule_cost)

    for _ in range(generations):
        population.sort(key=schedule_cost)
        next_population = [population[0].copy(), population[1].copy()]

        while len(next_population) < population_size:
            parent_a = choose_parent()
            parent_b = choose_parent()

            # One-point crossover: take some start times from each parent.
            cut = random.randint(1, len(appliances) - 1)
            child = parent_a[:cut] + parent_b[cut:]

            # Mutation: randomly change some appliance start times.
            for i, appliance in enumerate(appliances):
                if random.random() < mutation_rate:
                    child[i] = random.choice(appliance["start_hours"])

            next_population.append(child)

        population = next_population
        history.append(schedule_cost(population[0]))

    best_schedule = min(population, key=schedule_cost)
    return best_schedule, schedule_cost(best_schedule), history


ga_schedule, ga_score, ga_history = genetic_algorithm()
print("Genetic Algorithm cost score:", round(ga_score, 2))
print("Valid schedule:", is_valid(ga_schedule))
display(show_schedule(ga_schedule))

plt.figure(figsize=(9, 4))
plt.plot(ga_history)
plt.xlabel("Generation")
plt.ylabel("Best score")
plt.title("Genetic Algorithm progress")
plt.tight_layout()
plt.show()

## 8. Compare the algorithms

The table reports the real electricity cost separately from the penalty-based score. A schedule only counts as feasible if the time windows and load limit are satisfied.

In [ ]:
methods = [
    ("Baseline", baseline),
    ("Hill Climbing", hc_schedule),
    ("Simulated Annealing", sa_schedule),
    ("Genetic Algorithm", ga_schedule)
]

comparison_rows = []
for name, schedule in methods:
    real_cost = sum(
        appliance_cost(appliance, start)
        for appliance, start in zip(appliances, schedule)
    )
    comparison_rows.append({
        "Method": name,
        "Electricity cost": round(real_cost, 2),
        "Optimization score": round(schedule_cost(schedule), 2),
        "Valid": is_valid(schedule)
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

plt.figure(figsize=(9, 4))
plt.bar(comparison["Method"], comparison["Electricity cost"])
plt.ylabel("Electricity cost (example units)")
plt.title("Cost comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

valid_methods = [(name, schedule) for name, schedule in methods if is_valid(schedule)]
if valid_methods:
    best_name, best_schedule = min(
        valid_methods,
        key=lambda item: sum(
            appliance_cost(appliance, start)
            for appliance, start in zip(appliances, item[1])
        )
    )
    best_cost = sum(
        appliance_cost(appliance, start)
        for appliance, start in zip(appliances, best_schedule)
    )
    saving = baseline_real_cost - best_cost
    print("Best valid method:", best_name)
    print("Best valid cost:", round(best_cost, 2))
    print("Saving compared with baseline:", round(saving, 2))
else:
    best_name, best_schedule = min(methods, key=lambda item: schedule_cost(item[1]))
    print("No fully valid schedule was found in this run.")
    print("Best score found:", best_name)

## 9. Visualize the selected timetable

The Gantt chart shows each appliance's operating period. The load chart checks whether the total running power stays under the household limit.

In [ ]:
# Gantt chart
plt.figure(figsize=(11, 5))
for i, (appliance, start) in enumerate(zip(appliances, best_schedule)):
    plt.barh(i, appliance["duration"], left=start, height=0.6)
    plt.text(start + appliance["duration"] / 2, i, appliance["name"],
             ha="center", va="center", fontsize=8)

plt.yticks(range(len(appliances)), [a["name"] for a in appliances])
plt.xticks(range(0, 25, 2))
plt.xlim(0, 24)
plt.xlabel("Hour of day")
plt.title("Optimized appliance timetable")
plt.tight_layout()
plt.show()

# Total power being used in each hour
load = load_by_hour(best_schedule)
plt.figure(figsize=(11, 4))
plt.bar(hours, load)
plt.axhline(max_load, linestyle="--", label=f"Load limit: {max_load} kW")
plt.xticks(range(0, 24, 2))
plt.xlabel("Hour of day")
plt.ylabel("Combined load (kW)")
plt.title("Household load by hour")
plt.legend()
plt.tight_layout()
plt.show()

print("Chosen method:", best_name)
print("Schedule valid:", is_valid(best_schedule))
print("Peak load:", round(max(load), 2), "kW")
display(show_schedule(best_schedule))

## 10. Syllabus concepts and conclusion

This notebook implements the three main bio-inspired optimization methods directly. The syllabus also mentions ANN and supervised, unsupervised and reinforcement learning; these are different learning approaches, not required for the core cost-minimization algorithm itself.

For a short hackathon implementation, the strongest extension would be:
- **Supervised learning / ANN:** predict tomorrow's hourly price from real historical price data.
- **Unsupervised learning:** group hours into low, medium and high price periods.
- **Reinforcement learning:** learn when to run a flexible appliance from rewards based on electricity cost.

Do not claim these learning models are implemented in this notebook unless you add and test those sections. The current working prototype focuses on scheduling, cost optimization, constraints and visualization.

## 10. Extra syllabus concepts: ANN and supervised learning

For a short demonstration, we train a small neural network to learn the example hourly price pattern. Since we do not have real historical tariff records, we create synthetic training examples by adding small noise to the sample prices. This shows the technique, but it is not a real-world forecast.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# Create synthetic hourly observations across many days.
rng = np.random.default_rng(42)
training_hours = np.tile(np.arange(24), 80)
training_prices = np.tile(price, 80) + rng.normal(0, 0.25, len(training_hours))

# Use sine and cosine so that hour 23 is close to hour 0.
features = np.column_stack([
    np.sin(2 * np.pi * training_hours / 24),
    np.cos(2 * np.pi * training_hours / 24)
])

split_at = 60 * 24
ann = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(16,), max_iter=1000,
                 random_state=42, early_stopping=True)
)
ann.fit(features[:split_at], training_prices[:split_at])

predicted = ann.predict(features[split_at:])
print("ANN mean absolute error on synthetic test data:",
      round(mean_absolute_error(training_prices[split_at:], predicted), 3))

today_features = np.column_stack([
    np.sin(2 * np.pi * np.arange(24) / 24),
    np.cos(2 * np.pi * np.arange(24) / 24)
])
predicted_prices = ann.predict(today_features)

plt.figure(figsize=(10, 4))
plt.plot(hours, price, marker="o", label="Example prices")
plt.plot(hours, predicted_prices, marker=".", label="ANN prediction")
plt.xlabel("Hour of day")
plt.ylabel("Price per kWh")
plt.title("ANN price prediction demo")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Unsupervised learning: group hours by price

K-Means groups the 24 hours into three price bands based on price similarity. The cluster names are assigned after sorting the clusters by average price.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(np.array(price).reshape(-1, 1))

average_price = {
    cluster: np.mean(np.array(price)[cluster_ids == cluster])
    for cluster in np.unique(cluster_ids)
}
sorted_clusters = sorted(average_price, key=average_price.get)
cluster_name = {
    sorted_clusters[0]: "Low price",
    sorted_clusters[1]: "Medium price",
    sorted_clusters[2]: "High price"
}
hour_groups = [cluster_name[c] for c in cluster_ids]

display(pd.DataFrame({
    "Hour": [f"{h:02d}:00" for h in hours],
    "Price": price,
    "Price group": hour_groups
}))

plt.figure(figsize=(10, 4))
for group in ["Low price", "Medium price", "High price"]:
    chosen = [i for i, label in enumerate(hour_groups) if label == group]
    plt.scatter(chosen, [price[i] for i in chosen], label=group)
plt.xlabel("Hour of day")
plt.ylabel("Price per kWh")
plt.title("K-Means grouping of hourly prices")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Reinforcement learning: a small Q-learning example

This small example teaches an agent to choose a start hour for the washing machine. The reward is the negative running cost, so lower cost gives a higher reward. This is a separate demonstration and does not replace the multi-appliance optimizer.

In [ ]:
# Possible actions are the allowed washing-machine start times.
washing_machine = appliances[0]
actions = washing_machine["start_hours"]
q_values = np.zeros((24, len(actions)))

learning_rate = 0.15
discount = 0.9
exploration = 0.2

for episode in range(2000):
    state = random.randrange(24)

    # Epsilon-greedy action choice: explore sometimes, otherwise use best known action.
    if random.random() < exploration:
        action_index = random.randrange(len(actions))
    else:
        action_index = int(np.argmax(q_values[state]))

    start_hour = actions[action_index]
    reward = -appliance_cost(washing_machine, start_hour)
    next_state = (state + 1) % 24

    q_values[state, action_index] += learning_rate * (
        reward + discount * np.max(q_values[next_state])
        - q_values[state, action_index]
    )

learned_start = actions[int(np.argmax(q_values[0]))]
print(f"Q-learning suggests starting the washing machine at {learned_start:02d}:00.")
print("Its cost under the known example tariff:",
      round(appliance_cost(washing_machine, learned_start), 2))
print("Because the example tariff is already known, directly checking all allowed hours is also possible.")

## 13. Final takeaway

- **Hill Climbing, Simulated Annealing and Genetic Algorithm** search for a low-cost appliance schedule.
- **ANN / supervised learning** demonstrates predicting a price pattern from labelled examples.
- **K-Means / unsupervised learning** groups hours by price similarity.
- **Q-learning / reinforcement learning** demonstrates selecting an action using a reward.

The learning sections use synthetic or simplified examples. For a real smart-home system, use real tariff history, validate predictions on future dates, and strengthen constraint handling before controlling actual appliances.